Using Macro F1 as primary metric, along with weighted F1 and Top 3 accuracy 

Also include macro roc auc , macro pr auc, and log loss

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, LabelBinarizer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    average_precision_score,
    roc_auc_score,
    top_k_accuracy_score,
    log_loss
)

from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
def evaluate_model(y_true, y_prob):
    total_classes = y_prob.shape[1]

    # Derive class predictions from highest probability
    y_pred = y_prob.argmax(axis=1)
    
    # Binarize true labels for multiclass PR-AUC calculations
    lb = LabelBinarizer()
    lb.fit(range(total_classes))  # Ensure all classes are considered
    y_true_binarized = lb.transform(y_true)
    
    # Calculate metrics
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Identify classes that contain BOTH positive and negative examples in y_true
    # This prevents Scikit-Learn from crashing if a rare class lacks variance
    valid_auc_classes = (y_true_binarized.sum(axis=0) > 0) & (y_true_binarized.sum(axis=0) < len(y_true))
    
    if np.all(valid_auc_classes):
        macro_prauc = average_precision_score(y_true_binarized, y_prob, average='macro')
        macro_rocauc = roc_auc_score(y_true_binarized, y_prob, multi_class='ovr', average='macro')
    else:
        # Calculate macro metrics only over classes that have valid target distributions
        macro_prauc = average_precision_score(y_true_binarized[:, valid_auc_classes], y_prob[:, valid_auc_classes], average='macro')
        macro_rocauc = roc_auc_score(y_true_binarized[:, valid_auc_classes], y_prob[:, valid_auc_classes], multi_class='ovr', average='macro')
    
    
    top_3_acc = top_k_accuracy_score(y_true, y_prob, k=3, labels=range(total_classes))
    logloss = log_loss(y_true, y_prob, labels=range(total_classes))
    
    return {
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Macro PR-AUC": macro_prauc,
        "Macro ROC AUC": macro_rocauc,
        "Top-3 Accuracy": top_3_acc,
        "Log-Loss": logloss
    }

Read in the datasets

In [4]:
X_train = pd.read_csv('data/scaled_X_train.csv')
X_test = pd.read_csv('data/scaled_X_test.csv')
y_train = pd.read_csv('data/y_train.csv')
y_test = pd.read_csv('data/y_test.csv')

In [5]:
y_train['main_genre'].value_counts()

main_genre
2    769102
9    685703
7    276846
5    244855
0    239686
6    212991
4    181843
8     95952
3     83436
1     69972
Name: count, dtype: int64

In [ ]:
from sklearn.metrics import confusion_matrix

# Compute sample weights dynamically
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

# All models configured to address the class imbalance
models = {
    "Random Forest": RandomForestClassifier(
        class_weight='balanced_subsample', 
        n_jobs=-1, 
        random_state=42
    ),
    
    "LightGBM": lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(set(y_train)) ,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    
    "XGBoost": XGBClassifier(
        objective='multi:softprob', 
        num_class=len(set(y_train)) , 
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    ),
    
    "CatBoost": CatBoostClassifier(
        loss_function='MultiClass', 
        auto_class_weights='Balanced',
        random_state=42,
        verbose=50  
    )
}

performance_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    if name == "XGBoost":
        model.fit(X_train, y_train, sample_weight=sample_weights_train)
    elif name == "LightGBM":
        
        model.fit(
            X_train, y_train,
            sample_weight=sample_weights_train, 
            eval_set=[(X_test, y_test)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]  # Modern API format
        )

    elif name == "CatBoost":
        catboost_model = model
        catboost_model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train)


    # added to display confusion matrix
    if name == "CatBoost":
        y_t = y_test
        y_p = catboost_model.predict(X_test)

        matrix = confusion_matrix(y_t, y_p.ravel())


        # Define class labels in the correct order
        class_labels = ['Classical', 'Country', 'Electronic', 'Folk', 'Hip-Hop/R&B', 'Jazz/Blues', 'Metal', 'Pop', 'Punk', 'Rock'] 

        matrix_df = pd.DataFrame(
            matrix, 
            index=[f"Actual {label}" for label in class_labels], 
            columns=[f"Predicted {label}" for label in class_labels]
        )

        matrix_df

    print(f"Evaluating {name} on validation data...")

    y_prob_test = model.predict_proba(X_test)
    
    # Execute the evaluation function
    metrics = evaluate_model(y_test, y_prob_test)
    performance_results[name] = metrics

print("\n" + "="*50)
print("FINAL MODEL COMPARISON RESULTS")
print("="*50)

df_results = pd.DataFrame(performance_results).T
print(df_results.to_string(formatters={c: '{:,.4f}'.format for c in df_results.columns}))

print(matrix_df)


Training Random Forest...


c:\Users\627700\Documents\Music1\.venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Evaluating Random Forest on validation data...

Training LightGBM...


c:\Users\627700\Documents\Music1\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\627700\Documents\Music1\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\627700\Documents\Music1\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


Evaluating LightGBM on validation data...

Training XGBoost...
Evaluating XGBoost on validation data...

Training CatBoost...
Learning rate set to 0.118741
0:	learn: 2.1676416	total: 958ms	remaining: 15m 57s
50:	learn: 1.6078352	total: 35.2s	remaining: 10m 55s
100:	learn: 1.5693396	total: 1m 8s	remaining: 10m 8s
150:	learn: 1.5490044	total: 1m 42s	remaining: 9m 33s
200:	learn: 1.5348572	total: 2m 15s	remaining: 9m
250:	learn: 1.5249331	total: 2m 49s	remaining: 8m 25s
300:	learn: 1.5170741	total: 3m 22s	remaining: 7m 50s
350:	learn: 1.5104813	total: 3m 55s	remaining: 7m 15s
400:	learn: 1.5049031	total: 4m 28s	remaining: 6m 40s
450:	learn: 1.5003938	total: 5m 1s	remaining: 6m 6s
500:	learn: 1.4961463	total: 5m 33s	remaining: 5m 32s
550:	learn: 1.4925114	total: 6m 6s	remaining: 4m 58s
600:	learn: 1.4889402	total: 6m 38s	remaining: 4m 24s
650:	learn: 1.4857436	total: 7m 14s	remaining: 3m 52s
700:	learn: 1.4825984	total: 7m 48s	remaining: 3m 19s
750:	learn: 1.4797805	total: 8m 20s	remaining

Confusion matrix

In [7]:
matrix_df

,Predicted Classical,Predicted Country,Predicted Electronic,Predicted Folk,Predicted Hip-Hop/R&B,Predicted Jazz/Blues,Predicted Metal,Predicted Pop,Predicted Punk,Predicted Rock
Actual Classical,45452,1131,2813,3425,522,3857,1002,707,284,728
Actual Country,653,9989,236,2378,377,1332,102,1265,316,845
Actual Electronic,12840,3411,89415,7249,30487,11485,12116,13401,5796,6075
Actual Folk,1925,4738,936,6664,840,2379,422,1355,543,1057
Actual Hip-Hop/R&B,615,1299,5424,1666,28976,1966,636,2775,1138,966
Actual Jazz/Blues,6925,5020,3536,5969,3799,29025,742,2754,1050,2394
Actual Metal,1214,308,2730,817,610,597,35564,864,7645,2899
Actual Pop,2748,10966,6881,7393,8009,5103,2005,16477,4069,5561
Actual Punk,283,835,1391,761,1108,554,4669,874,11210,2303
Actual Rock,7089,21354,10223,14375,8229,11803,25424,15005,28549,29375


In [15]:
matrix_df.to_csv('data/2nd_confusion_matrix.csv', index=True)

Calculate feature importance

In [ ]:
importances = catboost_model.get_feature_importance()

# Map to feature names and sort
fi_df = pd.DataFrame({
    'feature_names': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(fi_df)

                           feature_names  importance
7                         mfcc_zero_mean   11.965654
4                             onset_rate   11.771657
5                       average_loudness    7.061485
3                           danceability    6.949206
23                    voice_instrumental    6.611478
16                       mood_aggressive    6.143683
18                       mood_electronic    5.780498
9        tuning_equal_tempered_deviation    5.104567
0                                    bpm    3.990520
8                       tuning_frequency    3.783705
17                         mood_acoustic    3.555240
21                                timbre    3.396220
6                     dynamic_complexity    3.385161
12                  mood_aggressive_prob    3.276941
13                            mood_happy    2.714711
14                              mood_sad    1.953156
11                       mood_happy_prob    1.800792
22                          tonal_atonal    1.

In [ ]:
#                            feature_names  importance
# 7                         mfcc_zero_mean   11.965654
# 4                             onset_rate   11.771657
# 5                       average_loudness    7.061485
# 3                           danceability    6.949206
# 23                    voice_instrumental    6.611478
# 16                       mood_aggressive    6.143683
# 18                       mood_electronic    5.780498
# 9        tuning_equal_tempered_deviation    5.104567
# 0                                    bpm    3.990520
# 8                       tuning_frequency    3.783705
# 17                         mood_acoustic    3.555240
# 21                                timbre    3.396220
# 6                     dynamic_complexity    3.385161
# 12                  mood_aggressive_prob    3.276941
# 13                            mood_happy    2.714711
# 14                              mood_sad    1.953156
# 11                       mood_happy_prob    1.800792
# 22                          tonal_atonal    1.641655
# 10                             key_scale    1.603916
# 20                          voice_gender    1.417150
# 19                            mood_party    1.010308
# 15                          mood_relaxed    0.786298
# 1     bpm_histogram_second_peak_bpm_mean    0.760220
# 25                            key_key_A#    0.476063
# 31                             key_key_E    0.323364
# 29                             key_key_D    0.315868
# 32                             key_key_F    0.291185
# 26                             key_key_B    0.290793
# 33                            key_key_F#    0.262269
# 24                             key_key_A    0.249856
# 30                            key_key_D#    0.237616
# 35                            key_key_G#    0.237362
# 27                             key_key_C    0.231659
# 28                            key_key_C#    0.215823
# 34                             key_key_G    0.202759
# 2   bpm_histogram_second_peak_bpm_median    0.201161

Save as pickle file